# Phase 14 — Grounded Generative RAG: Live Smoke Test

Single optional cell to call a locally running FastAPI server. Not a benchmark, and not a human quality evaluation. It does not download/retrain models by itself; the API may lazily fetch weights when invoked.


In [ ]:
# Phase 14 — single-cell, opt-in live smoke test. Requires a running local FastAPI server.
# This does NOT rebuild indexes or rerun any historical benchmark.
import json
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

BASE_URL = "http://127.0.0.1:8000"
QUESTION = "Ποιες αποζημιώσεις δόθηκαν στους αγρότες μετά τις πλημμύρες;"


def post_json(path, payload, timeout=900):
    body = json.dumps(payload, ensure_ascii=False).encode("utf-8")
    request = Request(BASE_URL + path, data=body,
                      headers={"Content-Type": "application/json"}, method="POST")
    with urlopen(request, timeout=timeout) as response:
        return response.status, json.loads(response.read().decode("utf-8"))

try:
    with urlopen(BASE_URL + "/health", timeout=10) as resp:
        print("Health:", resp.status, resp.read().decode("utf-8"))
    status, result = post_json("/rag/ask", {"question": QUESTION, "k": 3,
                                            "max_new_tokens": 96})
    print("RAG status:", status)
    print("Mode:", result.get("mode"))
    print("Abstained:", result.get("abstained"), "Reason:", result.get("reason"))
    print("Answer:", result.get("answer"))
    print("Faithfulness independently verified:", result.get("faithfulness_verified"))
    for source in result.get("sources", []):
        print(source["reference"], source["doc_id"], "|", source["excerpt"])
    assert status == 200 and result["mode"] == "grounded_rag"
    assert result["faithfulness_verified"] is False
    assert result["abstained"] or (result["answer"] and result["sources"])
    print("LIVE WIRING CHECK PASSED — answer quality still needs human evaluation.")
except (HTTPError, URLError, TimeoutError) as err:
    print("Live smoke test could not complete:", err)
    print("Check that the server is running, the index is available, and RAM is sufficient.")
    raise
